# Regression metrics for delivery estimates

At noon, Asha’s courier dispatches an order. The app estimates 25 minutes until arrival. The delivery actually takes 20 minutes. How do we measure that miss, and how do we combine misses across orders?

We’ll trace one residual, compare two ways errors can be spread, and read R² against a mean-prediction reference. These are the same five original synthetic orders used in the ML Learning Lab. Each row describes an elapsed time from dispatch in minutes.

In Colab, choose **Runtime → Run all** before teaching. Open the visual cards as you reach them. To teach live, edit just one prediction in **A short Python run** and rerun that cell. No files, installs, or credentials are needed.

In [ ]:
from html import escape
from math import isclose, sqrt
from statistics import mean
from IPython.display import HTML, display

# Frozen examples from data/delivery_metrics.json.
orders = ['E01', 'E02', 'E03', 'E04', 'E05']
actual = [20, 30, 40, 50, 60]
forecast_sets = {
    'Small misses': [25, 35, 45, 55, 65],
    'One large miss': [20, 30, 40, 50, 85],
    'Mean reference': [40, 40, 40, 40, 40],
    'Always predict 60 minutes': [60, 60, 60, 60, 60],
    'Perfect predictions': [20, 30, 40, 50, 60],
}

## A short Python run

Pair each forecast with its actual arrival. The **residual** is actual minus prediction, so a negative residual means the estimate was too high. Run the cell once. Then change just the first prediction from 25 to 20 and compare MAE and RMSE.

In [ ]:
predicted = forecast_sets['Small misses'].copy()
predicted[0] = 25  # Change this line to try another ETA for E01.
residuals = [observed - estimate for observed, estimate in zip(actual, predicted)]
mae = mean(abs(error) for error in residuals)
mse = mean(error**2 for error in residuals)
print(f'E01 residual = {actual[0]} − {predicted[0]} = {residuals[0]} min')
print(f'MAE = {mae:.2f} min | MSE = {mse:.2f} min² | RMSE = {sqrt(mse):.2f} min')

In [ ]:
# @title Load the visual helpers { display-mode: "form" }
panel_css = '''
<style>
  .rg-nb { padding:1rem 1.2rem; border:1px solid #cbded8; border-radius:12px; background:#f5faf8;
    color:#173a34; font:16px/1.5 system-ui,sans-serif; }
  .rg-nb h3,.rg-nb h4 { margin:.15rem 0 .55rem; }
  .rg-nb p { margin:.45rem 0 .75rem; }
  .rg-nb details { margin:.55rem 0; border:1px solid #d1e1db; border-radius:7px; background:white; }
  .rg-nb summary { padding:.5rem .7rem; cursor:pointer; font-weight:650; color:#12473f; }
  .rg-nb summary:focus-visible { outline:2px solid #b85f44; outline-offset:2px; }
  .rg-nb .inside { padding:0 .75rem .65rem; }
  .rg-nb .trace { margin:1rem 7px .35rem; height:8px; border-radius:6px; position:relative; background:#dceae5; }
  .rg-nb .trace i { position:absolute; top:50%; width:12px; height:12px; transform:translate(-50%,-50%);
    border-radius:50%; background:#147564; }
  .rg-nb .trace i+ i { border:2px solid #ab5039; border-radius:2px; background:white; }
  .rg-nb .axis { display:flex; justify-content:space-between; color:#3c6258; font-size:14px; }
  .rg-nb .bars { display:grid; grid-template-columns:minmax(4.6rem, max-content) minmax(1rem,1fr) max-content;
    gap:.3rem .55rem; align-items:center; max-width:42rem; }
  .rg-nb .track { height:10px; border-radius:6px; background:#dceae5; overflow:hidden; }
  .rg-nb .track > span { display:block; height:100%; background:#147564; }
  .rg-nb .track.reference > span { background:#3f6380; }
  .rg-nb .cards { display:grid; grid-template-columns:repeat(auto-fit,minmax(min(100%,16rem),1fr)); gap:.8rem; }
  .rg-nb article { padding:.6rem; border:1px solid #d1e1db; border-radius:6px; }
  .rg-nb .scale { font-size:14px; color:#3c6258; }
</style>'''

def display_panel(title, description, contents):
    display(HTML(panel_css + f'<section class="rg-nb"><h3>{escape(title)}</h3>'
                 f'<p>{escape(description)}</p>{contents}</section>'))

def disclosure(group, label, content, opened=False):
    state = ' open' if opened else ''
    return (f'<details name="{escape(group)}"{state}><summary>{escape(label)}</summary>'
            f'<div class="inside">{content}</div></details>')

def bar(label, number, maximum, unit, reference=False):
    variant = ' reference' if reference else ''
    return (f'<span>{escape(label)}</span><span class="track{variant}" aria-hidden="true">'
            f'<span style="width:{100 * number / maximum:.5f}%"></span></span>'
            f'<strong>{number:g} {escape(unit)}</strong>')

def error_metrics(observed, estimates):
    if not observed or len(observed) != len(estimates):
        raise ValueError('Supply one prediction for every observed delivery.')
    errors = [value - prediction for value, prediction in zip(observed, estimates)]
    target_mean = mean(observed)
    sse = sum(error**2 for error in errors)
    sst = sum((value - target_mean)**2 for value in observed)
    mse = sse / len(errors)
    return {
        'residuals': errors, 'mean_residual': mean(errors),
        'mae': mean(abs(error) for error in errors), 'mse': mse, 'rmse': sqrt(mse),
        'sse': sse, 'sst': sst, 'r2': None if sst == 0 else 1 - sse / sst,
    }

## Trace the forecast and arrival

The time estimate is made at dispatch. The arrival has to happen before we observe the target. Open an order and connect its two marks to its residual.

In [ ]:
# @title Open the delivery timeline { display-mode: "form" }
steady = forecast_sets['Small misses']
timeline_max = 70
order_cards = []
for index, (case, observed, estimate) in enumerate(zip(orders, actual, steady)):
    error = observed - estimate
    timeline = (f'<div class="trace" aria-hidden="true"><i style="left:{100 * observed/timeline_max:g}%"></i>'
                f'<i style="left:{100 * estimate/timeline_max:g}%"></i></div>'
                f'<div class="axis"><span>Dispatch · 0 min</span><span>{timeline_max} min</span></div>')
    reading = (f'<p><strong>Actual circle</strong> {observed} min · '
               f'<strong>Forecast square</strong> {estimate} min</p>'
               f'<p><strong>Residual = {observed} − {estimate} = {error} min</strong>. '
               'This estimate is too high.</p>')
    order_cards.append(disclosure('regression-orders', case, timeline + reading, opened=index == 0))
display_panel('Follow one order', 'Open a delivery to see its elapsed time and signed miss.', ''.join(order_cards))

## Compare error size

**MAE** averages absolute residuals in minutes. **MSE** averages squared residuals in minutes². **RMSE** takes the square root to return to minutes.

The comparison keeps the observed arrivals fixed. One set of predictions misses every order by five minutes. The other gets four orders right and misses the last by 25 minutes. The visual switches between absolute and squared errors. Within each view, both prediction sets use the same scale.

In [ ]:
# @title Compare absolute and squared per-order errors { display-mode: "form" }
def compare_errors(squared):
    names = ('Small misses', 'One large miss')
    results = [error_metrics(actual, forecast_sets[name]) for name in names]
    magnitudes = [[error**2 if squared else abs(error) for error in m['residuals']] for m in results]
    maximum = max(max(group) for group in magnitudes)
    unit = 'min²' if squared else 'min'
    cards = []
    for name, errors, result in zip(names, magnitudes, results):
        bars = ''.join(bar(case, value, maximum, unit) for case, value in zip(orders, errors))
        total = sum(errors)
        equation = (f'MSE = {total:g} ÷ {len(orders)} = {result["mse"]:g} min². '
                    f'RMSE = {result["rmse"]:.3f} min.' if squared else
                    f'MAE = {total:g} ÷ {len(orders)} = {result["mae"]:g} min.')
        cards.append(f'<article><h4>{escape(name)}</h4><div class="bars">{bars}</div>'
                     f'<p><strong>{equation}</strong></p></article>')
    return (f'<div class="cards">{"".join(cards)}</div>'
            f'<p class="scale">Shared scale from 0 to {maximum:g} {unit} per order.</p>')

display_panel('Same absolute error, different squared error',
              'Open a measure and compare its two sets of bars.',
              disclosure('regression-error', 'Absolute error', compare_errors(False), opened=True)
              + disclosure('regression-error', 'Squared error', compare_errors(True)))

In [ ]:
cancellation = actual.copy()
cancellation[0] -= 5
cancellation[1] += 5
result = error_metrics(actual, cancellation)
print('Residuals in minutes', result['residuals'])
print(f'Mean residual = {result["mean_residual"]:g} min | MAE = {result["mae"]:g} min')
assert result['mean_residual'] == 0 and result['mae'] == 2

One early forecast and one late forecast cancel in the mean residual. This is why average direction and typical error size answer different questions. If a customer wants to know how far off an estimate usually is, point to MAE.

## Read R² against the same reference

The observed deliveries average **40 minutes**. Measure how much squared error a model saves compared with predicting that same 40-minute reference for each delivery.

**R² = 1 − SSE ÷ SST**, where **SSE** sums the model’s squared residuals and **SST** sums the squared distances from the evaluation-target mean. R² is unitless. A value of 1 means exact predictions. Zero ties the reference. Negative means greater squared error than the reference.

In [ ]:
# @title Compare the model’s squared error and the observed-mean reference { display-mode: "form" }
results = {name: error_metrics(actual, prediction) for name, prediction in forecast_sets.items()}
scale = max(max(item['sse'] for item in results.values()), results['Small misses']['sst'])
reference_cards = []
for index, (name, result) in enumerate(results.items()):
    sse, sst, r2 = (result[key] for key in ('sse', 'sst', 'r2'))
    contents = ('<div class="bars">' + bar('Model SSE', sse, scale, 'min²')
                + bar('Reference SST', sst, scale, 'min²', reference=True)
                + f'</div><p><strong>R² = 1 − {sse:g} ÷ {sst:g} = {r2:g}</strong></p>')
    reference_cards.append(disclosure('regression-reference', name, contents, opened=index == 0))
display_panel('Model versus reference', f'All bars keep the same scale from 0 to {scale:g} min².',
              ''.join(reference_cards))

Standard R² uses the observed evaluation-target mean in its formula. Fit a real deployable constant predictor on training data before evaluating it on held-out deliveries. These are separate comparisons.

If all observed durations are identical, SST is zero and the usual R² ratio is undefined. A tool can display a finite convention. You can still report MAE and RMSE on such a batch. To compare models, use the same deliveries, units, and target definition.

In [ ]:
assert results['Small misses']['mae'] == results['One large miss']['mae'] == 5
assert results['Small misses']['sse'] == 125
assert results['One large miss']['sse'] == 625
assert isclose(results['One large miss']['rmse'], sqrt(125))
assert [item['r2'] for item in results.values()] == [0.875, 0.375, 0, -2, 1]
constant = error_metrics([40, 40, 40], [35, 40, 45])
assert constant['r2'] is None and constant['sst'] == 0
print(f'Constant-target example   R² undefined | MAE = {constant["mae"]:.3f} min')
print('All five delivery checkpoints passed.')

## Choose and explain the metric

Use MAE for a simple average miss in minutes. Use RMSE when large misses matter more and you want an answer in minutes. Use mean residual to check direction and R² to compare squared error with the mean reference. When the job has an explicit cost for late deliveries or very large misses, state that cost and measure it alongside the statistical metrics.